# Chapter 16 — Exploratory Data Analysis and Data Quality

Exploratory data analysis (EDA) combines data contracts, missingness analysis, outlier investigation, distributions, relationships, and communication. It turns a raw dataset into a defensible analytical starting point.

## Prerequisites
Pandas, NumPy, basic plotting, and introductory statistics.

## Learning goals
- Define grain, keys, and quality rules before analyzing.
- Profile missingness, duplicates, types, ranges, and categories.
- Investigate outliers without blindly deleting them.
- Explore univariate and multivariate structure.
- Create a reproducible EDA summary and communicate caveats.

**Study sequence:** use the theory notebook first, then tutorial, then lab. The examples use the bundled synthetic data unless otherwise noted.

In [1]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())

Data directory: /mnt/data/python_data_analysis_book/data


## Guided learning path

Work from top to bottom. Each step states the purpose, explains the API choice, and provides a complete code cell. Modify parameters and predict the output before running it.

### Step 1 — Guided 1 — State the grain and key

Before cleaning, decide what one row represents and which columns should identify it. Here order_id is expected to identify an order.

In [2]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())
import pandas as pd
df=pd.read_csv(DATA/"messy_sales.csv"); print("Candidate grain: one order row"); print("Rows:",len(df),"Unique order ids:",df.order_id.nunique())

Data directory: /mnt/data/python_data_analysis_book/data
Candidate grain: one order row
Rows: 62 Unique order ids: 60


### Step 2 — Guided 2 — Quantify missingness

Missingness rates help prioritize review, but cannot explain why a value is absent.

In [3]:
missing=df.isna().mean().sort_values(ascending=False); display(missing.to_frame("missing_rate"))

             missing_rate
region           0.112903
units            0.096774
order_id         0.000000
order_date       0.000000
product          0.000000
unit_price       0.000000
discount         0.000000
revenue          0.000000
cost             0.000000
profit           0.000000
customer_id      0.000000


### Step 3 — Guided 3 — Clean text and numeric fields

Normalize whitespace and capitalization, convert numeric columns explicitly, and count any values that fail conversion.

In [4]:
clean=df.copy(); clean["product"]=clean["product"].astype("string").str.strip().str.title(); clean["units"]=pd.to_numeric(clean.units,errors="coerce"); print("Missing units after conversion:",clean.units.isna().sum())

Missing units after conversion: 6


### Step 4 — Guided 4 — Review rather than auto-delete outliers

An IQR flag is a review queue. Keep valid high-value orders; correct or exclude only with documented evidence.

In [5]:
q1=clean.revenue.quantile(.25); q3=clean.revenue.quantile(.75); iqr=q3-q1; flagged=clean.loc[(clean.revenue<q1-1.5*iqr)|(clean.revenue>q3+1.5*iqr)]; print("Rows flagged for review:",len(flagged)); display(flagged.head())

Rows flagged for review: 13
   order_id  order_date region  ...       cost     profit  customer_id
3     O0005  2025-11-10   West  ...  130366.92   13956.47         C005
7     O0178  2025-05-07   East  ...  610328.12  115044.70         C013
13    O0058  2025-09-30   West  ...  608765.71  380965.94         C013
19    O0013  2025-06-22   East  ...  119465.54   65510.46         C013
22    O0189  2025-08-28    NaN  ...  458550.60  255604.02         C009

[5 rows x 11 columns]


## Tutorial checkpoint

1. What was the input data type?
2. What did the function or method return?
3. Which argument most changed the result?
4. What edge case should be tested before using this in production?